# 

In [ ]:
import pandas as pd
import numpy as np
column_names = ['symboling', 'normalized-losses', 'make', 'fuel-type', 'aspiration',
               'num-of-doors', 'body-style', 'drive-wheels', 'engine-location',
               'wheel-base', 'length', 'width', 'height', 'curb-weight',
               'engine-type', 'num-of-cylinders', 'engine-size', 'fuel-system',
               'bore', 'stroke', 'compression-ratio', 'horsepower', 'peak-rpm',
               'city-mpg', 'highway-mpg', 'price']

# Load data, recognizing '?' as NaN
data = pd.read_csv('datasets/automobile/automobile/imports-85.data', header=None, names=column_names, sep=',', na_values='?')

print("Automobile data loaded successfully!")
print("Initial shape:", data.shape)

In [ ]:
print("\nFirst 5 rows:")
print(data.head())

print("\nData Info:")
print(data.info())

print("\nNumerical Summary:")

print(data.describe(include='all'))

print("\nMissing Values per Column:")
print(data.isnull().sum())

print("\nValue Counts for 'body-style':")
print(data['body-style'].value_counts())

target_variable = 'price'
print(f"\nTarget Variable: {target_variable}")

In [ ]:
data.dropna(subset=[target_variable], axis=0, inplace=True)
print(f"\nDropped rows where target '{target_variable}' was missing. New shape: {data.shape}")

# 2. Impute numerical features with the mean
numerical_cols_to_impute = ['normalized-losses', 'bore', 'stroke', 'horsepower', 'peak-rpm']
for col in numerical_cols_to_impute:
    # Ensure column is numeric before imputing
    data[col] = pd.to_numeric(data[col], errors='coerce') # Should already be numeric if loaded correctly, but safe check
    mean_val = data[col].mean()
    data[col].fillna(mean_val, inplace=True)
    print(f"Filled NaNs in numerical '{col}' with mean: {mean_val:.2f}")

# 3. Impute categorical features with the mode
categorical_cols_to_impute = ['num-of-doors']
for col in categorical_cols_to_impute:
    mode_val = data[col].mode()[0]
    data[col].fillna(mode_val, inplace=True)
    print(f"Filled NaNs in categorical '{col}' with mode: {mode_val}")

# Verify no more missing values (except maybe ones you decided not to handle)
print("\nMissing Values after Imputation:")
print(data.isnull().sum()) # Should show 0 for imputed columns


# B) Convert Categorical Features to Numerical (One-Hot Encoding)
# Identify remaining categorical columns (object type)
categorical_cols = data.select_dtypes(include='object').columns
print(f"\nCategorical columns to encode: {list(categorical_cols)}")

# Apply one-hot encoding
data_encoded = pd.get_dummies(data, columns=categorical_cols, drop_first=True)

print("\nData after one-hot encoding (first 5 rows):")
print(data_encoded.head())
print("\nNew shape after encoding:", data_encoded.shape)


# C) Ensure Target is Numeric and Separate Features/Target
# Convert target 'price' to numeric (it might still be object if loaded with '?')
data_encoded[target_variable] = pd.to_numeric(data_encoded[target_variable], errors='coerce')

# Separate Features (X) and Target (y)
X = data_encoded.drop(target_variable, axis=1)
y = data_encoded[target_variable]

# Verify separation
print(f"\nFeatures (X) and Target (y) separated.")
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print("Data type of y:", y.dtype) # Should be numeric now


# D) Feature Scaling (To be done within Pipeline)
# Identify numerical feature columns in the final feature set X for the scaler
numerical_features = X.select_dtypes(include=np.number).columns.tolist()
print(f"\nNumerical features identified for scaling: {len(numerical_features)}")

# --- End of Pre-processing ---

In [ ]:
# --- Imports for Models and Evaluation ---
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression # For regression
from sklearn.svm import SVR # For regression
from sklearn.tree import DecisionTreeRegressor # For regression
from sklearn.neural_network import MLPRegressor # For regression

import numpy as np
import pandas as pd # Re-import if in a new script/session


print("\n--- Checking Data Types in X before Pipeline ---")
print(X.info())

K = 5 # Or K=10, your group decides
kf = KFold(n_splits=K, shuffle=True, random_state=42) # random_state for reproducibility
print(f"\nUsing K={K} for Cross-Validation.")

models_reg = {
    'Linear Regression': LinearRegression(),
    'SVR': SVR(), # Default kernel='rbf', C=1.0
    'Decision Tree Regressor': DecisionTreeRegressor(random_state=42), # No depth limit by default
    'MLP Regressor': MLPRegressor(random_state=42,
                                  max_iter=1000, # Increased iterations for convergence
                                  hidden_layer_sizes=(100,), # Default: one hidden layer with 100 neurons
                                  activation='relu', # Default activation
                                  solver='adam', # Default solver
                                  early_stopping=True, # Good practice to prevent overfitting
                                  n_iter_no_change=10) # Stops if validation score doesn't improve
}

# For the automobile 'price' prediction, it's regression
current_task = 'regression'
scoring_metric = 'neg_mean_squared_error' # Scikit-learn convention for MSE in CV
metric_name = 'Mean Squared Error (MSE)'
print(f"Task: {current_task}, Evaluation Metric: {metric_name}")


# --- Evaluation Loop using K-Fold CV ---
results = {} # Dictionary to store results

print(f"\n--- Running {K}-Fold Cross-Validation ---")

for model_name, model in models_reg.items():
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    scores = cross_val_score(pipeline, X, y, cv=kf, scoring=scoring_metric, n_jobs=-1) # n_jobs=-1 uses all CPU cores
    mean_score = -np.mean(scores)
    std_dev_score = np.std(scores) # Standard deviation of the scores across folds

    results[model_name] = {'Mean MSE': mean_score, 'Std Dev MSE': std_dev_score}

    print(f"{model_name}: Mean MSE = {mean_score:,.2f} (+/- {std_dev_score:,.2f})")

# --- Record Parameters (Example Notes for Report) ---

print("\n--- Parameter Notes (Example for Report) ---")
print(f"K-Fold: K={K}, shuffle=True, random_state=42")
print("Scaling: StandardScaler used within pipeline.")
print("Linear Regression: Default parameters.")
print(f"SVR: Default parameters (e.g., kernel='{SVR().kernel}', C={SVR().C}).")
print(f"Decision Tree Regressor: random_state=42, default parameters (e.g., criterion='{DecisionTreeRegressor().criterion}', max_depth=None).")
print(f"MLP Regressor: random_state=42, max_iter=1000, hidden_layer_sizes=(100,), activation='relu', solver='adam', early_stopping=True.")

# --- Convert results to DataFrame for easy reporting ---
results_df = pd.DataFrame(results).T # Transpose rows/columns
print("\n--- Summary Table for Report ---")
print(results_df.to_markdown()) # Print in Markdown format suitable for report